# Tres comparaciones para la estadificación del sueño

Sleep Cassette, cinco estadios y épocas de 30 s. El protocolo usa los cinco folds congelados de los 62 sujetos de desarrollo. La métrica primaria es F1 macro medio entre folds. Las secundarias incluyen exactitud, kappa por fold, precisión, sensibilidad y F1 de N1, y errores cerca de las transiciones. El recorte de vigilia de ±30 minutos usa anotaciones y las ventanas son retrospectivas.

| Combinación | Comparaciones | Pregunta |
|---|---|---|
| 1 | TCN y BiLSTM con ventanas centradas de 1 y 15 épocas, con las mismas 145 variables por época, escalador y sujetos. | ¿Qué aporta el contexto y qué aporta la arquitectura? |
| 2 | TCN sobre descriptores EEG/EOG, CNN de ondas + TCN, LightGBM de descriptores con contexto y mezcla fija 50/50. Cuando existe el artefacto, se incluye el LightGBM histórico de cuatro canales. | ¿Las ondas contienen información complementaria a los descriptores? |
| 3 | TCN, Transformer y el mismo Transformer con pérdida auxiliar de transición. Los tres leen los mismos embeddings de 15 épocas por fold y fase de entrenamiento. | ¿La atención o la supervisión de transiciones mejoran N1 y las épocas cercanas a un cambio? |

## Ejecución en MPS y CUDA

Desde la raíz del repositorio, instalar el entorno con `uv sync --locked --extra dl --extra boosting --group notebook`. La opción `auto` busca MPS y CUDA. Para elegir explícitamente, usar `--device mps` o `--device cuda`. En CUDA se necesita un controlador compatible con la compilación instalada de PyTorch. La comparación LightGBM usa su implementación nativa en un proceso CPU separado.

```sh
uv run --locked --extra dl --extra boosting python scripts/run_tcn_research.py --smoke --device auto
uv run --locked --extra dl --extra boosting python scripts/run_tcn_research.py --combination all --device auto
```

Para ejecutar un bloque, elegir `--combination 1`, `2` o `3`. La primera combinación solo necesita las variables de la campaña de desarrollo. En otra máquina, copiar `features.parquet` junto a su `metadata.json` y pasar `--features /ruta/features.parquet`. Las combinaciones 2 y 3 además necesitan `--source-root /ruta/al/proyecto-con-datos` o `SLEEP_RESEARCH_DATA_ROOT`. Esa raíz contiene `data/manifests/sleep-cassette/recordings.parquet` y los EDF de desarrollo en sus rutas originales. La caché de ondas completa ocupa unos 5.5 GB decimales. Para incluir el campeón previo, copiar el `lgbm.json` de la campaña de refinamiento junto con `runs/<identidad>/lgbm/oof.parquet` y pasar `--champion /ruta/lgbm.json`.

El modo `--smoke` usa fragmentos de una noche real por fold y una época de entrenamiento para comprobar el flujo. Sus tablas se guardan en `smoke/`. Una campaña científica genera `runs/<identidad>/leaderboard-<combinación>.csv`, predicciones por época, métricas y checkpoints. Al repetir el comando se reutilizan los folds neuronales completos y los embeddings compatibles. `--batch-size` permite reducir memoria y `--seed` permite repetir con otra semilla. El valor de la semilla también controla la partición interna.

## Entrenamiento y evaluación

Cada fold reserva el 20 % de sus sujetos de entrenamiento para elegir la época del clasificador por F1 macro. El refit arranca de cero usando todo el entrenamiento externo. Los dos EEG y el EOG conservan sus 100 Hz y se filtran por noche. Una CNN supervisada por el estadio de la época aprende un embedding de 64 dimensiones durante cinco épocas fijas. Se entrena un encoder interno y otro de refit por fold. La segunda etapa congela estos embeddings, de modo que cada clasificador temporal reciba la misma representación.

La TCN utiliza tres convoluciones con dilaciones 1, 2 y 4. La BiLSTM tiene 64 unidades por dirección. El Transformer usa dos bloques, cuatro cabezas y posiciones aprendidas. Los clasificadores emplean AdamW, tasa 0.001, dropout 0.2, entropía cruzada ponderada por la raíz de la frecuencia inversa, máximo de 20 épocas y paciencia de 4. La opción auxiliar añade entropía cruzada binaria con peso 0.2. Su objetivo vale uno cuando el estadio difiere del vecino anterior o posterior. Un borde, hueco o estadio desconocido excluye esa época de la pérdida auxiliar y del análisis de transiciones. La etiqueta auxiliar solo interviene en entrenamiento y evaluación.

LightGBM usa descriptores de los tres canales y sus resúmenes temporales retrospectivos. El número de árboles se elige con validación interna. La mezcla promedia sus probabilidades fuera de muestra con las del CNN–TCN después de verificar correspondencia por época, sujeto, registro, fold y etiqueta.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display
from sleep_staging.research import paired_subject_bootstrap

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/run_tcn_research.py").is_file())
OUTPUT = ROOT / "var/research/sc-tcn-v1"
COMBINATION = "all"
DEVICE = "auto"
RUN_TRAINING = False
# Fijar una ruta para revisar una campaña o semilla específica.
RUN_DIR = None

if RUN_TRAINING:
    subprocess.run([sys.executable, str(ROOT / "scripts/run_tcn_research.py"),
                    "--combination", COMBINATION, "--device", DEVICE, "--output", str(OUTPUT)],
                   cwd=ROOT, env=os.environ.copy(), check=True)

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Comparación de la campaña

Al abrir el notebook, la tabla recoge la campaña científica terminada más reciente de la combinación seleccionada. Para comparar distintas semillas, establecer explícitamente `RUN_DIR` en cada revisión. Los resultados completos por fold y por estadio se encuentran en `<modelo>/metrics.json`.

In [2]:
name = f"leaderboard-{COMBINATION}.csv"
tables = sorted((OUTPUT / "runs").glob(f"*/{name}"), key=lambda p: p.stat().st_mtime)
table = Path(RUN_DIR) / name if RUN_DIR else (tables[-1] if tables else None)
if table is None:
    print("Todavía no hay una campaña científica terminada para esta combinación.")
else:
    board = pd.read_csv(table).sort_values("macro_f1", ascending=False)
    print(table.relative_to(ROOT))
    display(board.round(4))
    winner = board.iloc[0]
    print(f"Mayor F1 macro de desarrollo: {winner.model}, {winner.macro_f1:.4f} ± {winner.fold_sd:.4f}.")

var/research/sc-tcn-v1/runs/6e398e69da0dee3125c3/leaderboard-all.csv


,model,macro_f1,fold_sd,n1_f1,n1_precision,n1_recall,accuracy,transition_accuracy,stable_accuracy
0,wave_tcn_lightgbm_50_50,0.8009,0.0183,0.5401,0.5253,0.5557,0.8429,0.6170,0.8914
1,historical_lightgbm_four_channels,0.7893,0.0179,0.5159,0.4787,0.5593,0.8300,0.6060,0.8780
2,matched_lightgbm,0.7888,0.0173,0.5127,0.4759,0.5556,0.8300,0.6039,0.8784
3,wave_cnn_tcn_15,0.7847,0.0233,0.5200,0.5139,0.5263,0.8289,0.6020,0.8777
4,features_tcn_15,0.7799,0.0220,0.5114,0.4722,0.5576,0.8222,0.5931,0.8720
5,wave_transformer_aux,0.7792,0.0207,0.5149,0.4952,0.5361,0.8221,0.5986,0.8699
6,features_bilstm_15,0.7791,0.0166,0.5174,0.4705,0.5746,0.8210,0.5943,0.8696
7,eeg_eog_features_tcn_15,0.7776,0.0228,0.5094,0.4583,0.5735,0.8200,0.5939,0.8686
8,wave_transformer,0.7756,0.0242,0.5052,0.4928,0.5183,0.8186,0.5969,0.8671
9,features_bilstm_1,0.7448,0.0228,0.4601,0.4047,0.5329,0.7889,0.6005,0.8288


Mayor F1 macro de desarrollo: wave_tcn_lightgbm_50_50, 0.8009 ± 0.0183.


## Contrastes emparejados por sujeto

La diferencia de F1 macro se calcula agrupando todas las predicciones externas. El intervalo remuestrea sujetos completos 2.000 veces y es condicional a los modelos ya entrenados. La tabla principal da igual peso a cada fold.

In [3]:
contrasts = [
    ("features_tcn_1", "features_tcn_15"),
    ("features_bilstm_15", "features_tcn_15"),
    ("eeg_eog_features_tcn_15", "wave_cnn_tcn_15"),
    ("matched_lightgbm", "wave_tcn_lightgbm_50_50"),
    ("wave_cnn_tcn_15", "wave_transformer"),
    ("wave_transformer", "wave_transformer_aux"),
]
if table is not None:
    rows = []
    for baseline, alternative in contrasts:
        if baseline not in set(board.model) or alternative not in set(board.model):
            continue
        frames = [pd.read_parquet(table.parent / key / "predictions.parquet") for key in (baseline, alternative)]
        paired = paired_subject_bootstrap(*frames)
        rows.append({"Referencia": baseline, "Alternativa": alternative, "Delta F1 agrupado": paired["delta"],
                     "IC inferior": paired["ci95"][0], "IC superior": paired["ci95"][1]})
    display(pd.DataFrame(rows).round(4))

,Referencia,Alternativa,Delta F1 agrupado,IC inferior,IC superior
0,features_tcn_1,features_tcn_15,0.0417,0.0333,0.0505
1,features_bilstm_15,features_tcn_15,0.0006,-0.0060,0.0074
2,eeg_eog_features_tcn_15,wave_cnn_tcn_15,0.0065,-0.0025,0.0166
3,matched_lightgbm,wave_tcn_lightgbm_50_50,0.0120,0.0083,0.0156
4,wave_cnn_tcn_15,wave_transformer,-0.0091,-0.0176,-0.0020
5,wave_transformer,wave_transformer_aux,0.0040,-0.0033,0.0119
